# Part 3 — Adversarial Attacks

Self-contained notebook. Re-runs the data load and (where needed) reloads the saved baseline checkpoint so it can execute standalone on Colab.

In [ ]:
!pip install -q transformers torch scikit-learn fairlearn aif360 pandas matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

USE_COLS = ["comment_text", "toxic", "black", "white", "muslim", "jewish"]
IDENTITY_COLS = ["black", "white", "muslim", "jewish"]

raw = pd.read_csv("data/jigsaw-unintended-bias-train.csv", usecols=USE_COLS)
raw["label"] = (raw["toxic"] >= 0.5).astype(int)

sample = raw.sample(n=120_000, random_state=SEED)
train_df, eval_df = train_test_split(
    sample, test_size=20_000, stratify=sample["label"], random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)
print("train:", train_df.shape, "eval:", eval_df.shape)


In [ ]:
import os, glob
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128

# Prefer best mitigated checkpoint if present (Part 5 scenario), else baseline,
# else fall back to the pretrained backbone.
_candidates = (sorted(glob.glob("checkpoints/best_mitigated_*"))
               + ["checkpoints/baseline"])
CKPT_DIR = next((c for c in _candidates if os.path.isdir(c)), MODEL_NAME)
print("loading model from", CKPT_DIR)

tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR)
model = AutoModelForSequenceClassification.from_pretrained(CKPT_DIR, num_labels=2)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

class ToxicDataset(Dataset):
    def __init__(self, df):
        enc = tokenizer(df["comment_text"].astype(str).tolist(),
                        truncation=True, padding="max_length", max_length=MAX_LEN)
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels = df["label"].tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {
            "input_ids": torch.tensor(self.input_ids[i]),
            "attention_mask": torch.tensor(self.attention_mask[i]),
            "labels": torch.tensor(self.labels[i]),
        }

train_ds = ToxicDataset(train_df)
eval_ds = ToxicDataset(eval_df)


In [ ]:
# Score the eval set so downstream cells can reuse predictions.
from transformers import Trainer, TrainingArguments

_args = TrainingArguments(output_dir="tmp_eval", per_device_eval_batch_size=64,
                          report_to="none")
_trainer = Trainer(model=model, args=_args)
_logits = _trainer.predict(eval_ds).predictions
eval_probs = torch.softmax(torch.tensor(_logits), dim=-1)[:, 1].numpy()
eval_preds = (eval_probs >= 0.5).astype(int)
eval_df = eval_df.assign(prob=eval_probs, pred=eval_preds)
trainer = _trainer  # downstream cells reference `trainer` and `metrics`
metrics = _trainer.evaluate(eval_ds)
print({k: round(v, 4) for k, v in metrics.items() if k.startswith("eval_")})


In [ ]:
# Shared helpers reused across multiple parts (defined once here so each
# notebook is fully standalone).
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, precision_score, recall_score)

def compute_metrics(pred):
    logits, labels = pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "auc_roc": roc_auc_score(labels, probs),
    }

def per_cohort_metrics(df):
    y, p = df["label"].values, df["pred"].values
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return {
        "n": len(df),
        "TPR": tp / (tp + fn) if (tp + fn) else 0.0,
        "FPR": fp / (fp + tn) if (fp + tn) else 0.0,
        "FNR": fn / (tp + fn) if (tp + fn) else 0.0,
        "Precision": precision_score(y, p, zero_division=0),
    }


In [ ]:
import random

ZWSP = "\u200b"  # zero-width space — invisible to humans, splits subword tokens
HOMOGLYPHS = {"a": "\u0430", "e": "\u0435", "o": "\u043e",
              "p": "\u0440", "c": "\u0441", "x": "\u0445"}

rng = random.Random(SEED)

def perturb(text: str) -> str:
    out = []
    for word in text.split(" "):
        chars = []
        for i, ch in enumerate(word):
            ch = HOMOGLYPHS.get(ch.lower(), ch)
            chars.append(ch)
            # 20% duplication rate per spec
            if rng.random() < 0.20:
                chars.append(ch)
            # ZWSP every 2-3 chars
            if i > 0 and i % rng.choice([2, 3]) == 0:
                chars.append(ZWSP)
        out.append("".join(chars))
    return " ".join(out)

print("orig:", "I hate you so much")
print("pert:", perturb("I hate you so much"))

In [ ]:
def predict_texts(texts, batch_size=64):
    model.eval()
    device = next(model.parameters()).device
    probs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = tokenizer(texts[i:i+batch_size], truncation=True,
                              padding=True, max_length=MAX_LEN, return_tensors="pt").to(device)
            logits = model(**batch).logits
            probs.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    return np.array(probs)

# Pick toxic comments the model is confidently calling toxic.
attack_pool = eval_df[(eval_df["pred"] == 1) & (eval_df["prob"] >= 0.7)]
attack_sample = attack_pool.sample(n=min(500, len(attack_pool)), random_state=SEED)
print(f"attacking {len(attack_sample)} confident-toxic comments")

orig_texts = attack_sample["comment_text"].astype(str).tolist()
pert_texts = [perturb(t) for t in orig_texts]

orig_probs = predict_texts(orig_texts)
pert_probs = predict_texts(pert_texts)

asr = float(np.mean(pert_probs < 0.5))
print(f"Attack Success Rate: {asr:.3f}")
print(f"avg confidence  before: {orig_probs.mean():.3f}  after: {pert_probs.mean():.3f}")

In [ ]:
# --- Poisoning attack: flip 5% of training labels and retrain from pretrained.
poison_df = train_df.copy()
flip_idx = poison_df.sample(frac=0.05, random_state=SEED).index
poison_df.loc[flip_idx, "label"] = 1 - poison_df.loc[flip_idx, "label"]
print(f"flipped {len(flip_idx)} labels ({len(flip_idx)/len(poison_df):.1%})")

poison_train_ds = ToxicDataset(poison_df)
poison_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

poison_args = TrainingArguments(
    output_dir="checkpoints/poisoned",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=200,
    seed=SEED,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

poison_trainer = Trainer(
    model=poison_model, args=poison_args,
    train_dataset=poison_train_ds, eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)
poison_trainer.train()

In [ ]:
def strip_eval(d):
    return {k.replace("eval_", ""): v for k, v in d.items() if k.startswith("eval_")}

compare = pd.DataFrame({
    "clean": strip_eval(metrics),
    "poisoned": strip_eval(poison_trainer.evaluate()),
}).round(4)
compare["delta"] = (compare["poisoned"] - compare["clean"]).round(4)
print(compare)

In [ ]:
# FNR is the most operationally important poisoning effect: a higher FNR
# means the poisoned model lets more genuinely toxic content through.
def fnr(y_true, y_pred):
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    pos = (y_true == 1).sum()
    return fn / pos if pos else 0.0

clean_pred = (eval_probs >= 0.5).astype(int)
poison_logits = poison_trainer.predict(eval_ds).predictions
poison_pred = (torch.softmax(torch.tensor(poison_logits), dim=-1)[:, 1].numpy() >= 0.5).astype(int)

fnr_table = pd.DataFrame({
    "clean": [fnr(eval_df["label"].values, clean_pred)],
    "poisoned": [fnr(eval_df["label"].values, poison_pred)],
}, index=["FNR"]).round(4)
fnr_table["delta"] = (fnr_table["poisoned"] - fnr_table["clean"]).round(4)
print(fnr_table)

### Which attack is more operationally dangerous?

**Evasion is the more realistic threat for a public social platform.** It needs nothing but the public model output: any user can iterate locally on perturbations until a comment scores < 0.5, and the cost per attempt is essentially free. The attack also scales — a single Tampermonkey script can apply `perturb()` automatically to any post.

**Poisoning is more catastrophic per success but assumes much more access.** The attacker needs to either compromise the labeling pipeline (insider threat or supply-chain attack on a crowdwork vendor) or push enough adversarial labels through user-feedback loops to materially shift a 100k-row training set. That's a different order of attacker capability.

**Where to invest defenses** (in order):

1. **Input normalization** for evasion — strip zero-width characters, NFKC-normalize Unicode to fold homoglyphs, collapse runs of duplicate characters. These three pre-processing steps zero out almost all of `perturb()`'s effect at near-zero latency cost.
2. **Calibration + human-in-the-loop** for the borderline band — exactly what the Section 6 pipeline does.
3. **Data provenance + label auditing** for poisoning — slower payoff but the only real defense: track who labeled what, sample-audit batches, monitor per-source label drift.